# Model Stress-Test & Failure Catalog 🔍💥

On Day 8, we built a model that predicts whether an order will be cancelled, and it looked pretty good — 87% recall on the cancelled class. That number is where most portfolios stop.

**This project doesn't trust that number. It goes looking for exactly where the model breaks, and writes down why.**

**Analogy:** imagine hiring a security guard and being told "they catch 87% of shoplifters." Great — but a good manager's NEXT question is "which 13% do they miss, and is there a PATTERN to who gets past them?" That's this entire notebook.

## 0. Rebuild the exact Day 8 model

Nothing new here on purpose — we need the *same* model to stress-test, not a fresh one.

In [ ]:
import pandas as pd
import numpy as np
from features import build_order_features
from model_utils import train_model, predict, NUM_FEATURES, CAT_FEATURES

df = pd.read_csv('data/raw/online_retail_II.csv', encoding='latin1')
orders = build_order_features(df)

bundle = train_model(orders)
print("Model trained. Test set size:", bundle['X_test'].shape[0])

In [ ]:
from sklearn.metrics import classification_report

y_pred, y_proba = predict(bundle, bundle['X_test'])
print(classification_report(bundle['y_test'], y_pred, target_names=['Not Cancelled','Cancelled'], digits=3))

Same result as Day 8: 87% recall on the Cancelled class. Now let's stop trusting that single number and go find the 13% it misses.

## 1. Pulling out the real mistakes

**Analogy:** instead of asking "what's my test score?", we're pulling out the actual wrong answers and spreading them on the table to look for a pattern — the way a teacher reviews wrong answers instead of just recording the grade.

Four groups exist for any yes/no prediction:
- **True Positive** — actually cancelled, model correctly caught it ✅
- **True Negative** — actually fine, model correctly said fine ✅
- **False Negative** — actually cancelled, model MISSED it (said "fine") 🚨 — the dangerous one
- **False Positive** — actually fine, model raised a false alarm ⚠️ — annoying, not dangerous

In [ ]:
results = bundle['X_test'].copy()
results['actual'] = bundle['y_test'].values
results['predicted'] = y_pred
results['confidence'] = y_proba

false_negatives = results[(results['actual']==1) & (results['predicted']==0)]
false_positives = results[(results['actual']==0) & (results['predicted']==1)]
true_positives  = results[(results['actual']==1) & (results['predicted']==1)]

print(f"False negatives (missed cancellations): {len(false_negatives)}")
print(f"False positives (false alarms):         {len(false_positives)}")
print(f"True positives (correctly caught):       {len(true_positives)}")

## 2. The pattern — comparing what the model CAUGHT vs. what it MISSED

Both groups below are genuinely cancelled orders. The only question: did the model catch it?

In [ ]:
compare_cols = ['num_unique_products','total_quantity','total_value','avg_unit_price']

print("Orders the model MISSED (median values):")
print(false_negatives[compare_cols].median())
print()
print("Orders the model CAUGHT (median values):")
print(true_positives[compare_cols].median())

**Here's the blind spot, in plain numbers:**

| | Missed cancellations | Caught cancellations |
|---|---|---|
| Median items in order | **4** | **1** |
| Median quantity | **41** | **3** |
| Median value | **£113.55** | **£14.93** |

**In plain words: the model has quietly learned "small order = risky, big order = safe."** It's genuinely good at catching tiny, 1-item cancellations — but a cancelled order that happens to be bigger or more varied slips right past it, specifically BECAUSE it looks big and "normal."

**Analogy:** imagine a bouncer who's learned "trouble always wears a red jacket" from experience — they'll catch every red jacket perfectly, but someone causing just as much trouble in a blue jacket walks right past, not because the bouncer is bad at their job, but because they latched onto one visible pattern instead of the real signal underneath.

## 3. How confidently wrong is the model — barely unsure, or completely fooled?

**Analogy:** there's a big difference between a student who says "hmm, 49% sure" and gets it wrong (an honest near-miss) versus one who says "I'm 95% sure the answer is B" and it's actually A (a confident, real misunderstanding). Let's find out which one this is.

In [ ]:
print("Confidence scores on the MISSED cancellations (should have been close to 1.0):")
print(false_negatives['confidence'].describe())

**The median confidence on missed cancellations is only ~0.20** — nowhere near the 0.5 cutoff. This isn't the model being narrowly unsure and just barely missing the call. It's **confidently telling us these orders are safe when they weren't** — the more worrying kind of failure, because a human reviewing "borderline" cases (say, 0.4–0.6 confidence) would never even see these — the model is too sure of itself to flag them for review.

## 4. Confirming it with made-up examples we can actually reason about

Real test-set rows prove the pattern exists. Now let's build a few hand-crafted "what if" orders to see the blind spot directly, in numbers anyone can sanity-check.

In [ ]:
synthetic_cases = pd.DataFrame([
    {"name": "Tiny single-item order",   "num_unique_products":1,  "total_quantity":2,   "total_value":8.50,  "avg_unit_price":4.25, "Hour":11, "Country":"United Kingdom", "DayOfWeek":"Tuesday"},
    {"name": "Large everyday bulk order","num_unique_products":18, "total_quantity":210, "total_value":540.00,"avg_unit_price":2.60, "Hour":13, "Country":"United Kingdom", "DayOfWeek":"Thursday"},
    {"name": "Medium multi-item order",  "num_unique_products":5,  "total_quantity":45,  "total_value":120.00,"avg_unit_price":2.70, "Hour":14, "Country":"United Kingdom", "DayOfWeek":"Wednesday"},
    {"name": "Rare-country small order", "num_unique_products":1,  "total_quantity":3,   "total_value":15.00, "avg_unit_price":5.00, "Hour":10, "Country":"Other",          "DayOfWeek":"Monday"},
    {"name": "Weekend large order",      "num_unique_products":20, "total_quantity":180, "total_value":610.00,"avg_unit_price":3.40, "Hour":12, "Country":"United Kingdom", "DayOfWeek":"Saturday"},
    {"name": "Zero-product edge case",   "num_unique_products":0,  "total_quantity":0,   "total_value":0.0,   "avg_unit_price":0.0,  "Hour":9,  "Country":"United Kingdom", "DayOfWeek":"Friday"},
])

pred_synth, proba_synth = predict(bundle, synthetic_cases)
synthetic_cases['predicted_cancel_probability'] = proba_synth
synthetic_cases['predicted_label'] = np.where(pred_synth==1, 'Cancelled', 'Not Cancelled')

synthetic_cases[['name','total_value','num_unique_products','predicted_cancel_probability','predicted_label']]

## 5. The failure catalog — every case, with a verdict

| Case | Model says | Confidence it's Cancelled | Verdict |
|---|---|---|---|
| Tiny single-item order (£8.50) | Cancelled | 92.6% | ✅ Correct instinct — matches the real "small orders are risky" pattern |
| **Large everyday bulk order (£540, 18 items)** | Not Cancelled | **2.8%** | 🚨 **The blind spot.** If this were actually cancelled, the model would be badly, confidently wrong |
| Medium multi-item order (£120, 5 items) | Not Cancelled | 6.6% | ⚠️ Same blind spot, milder — order size is already pulling it toward "safe" |
| Rare-country small order (£15) | Cancelled | 82.0% | ✅ Order size still dominates correctly, even from an underrepresented country |
| **Weekend large order (£610, 20 items)** | Not Cancelled | **3.5%** | 🚨 Same blind spot — weekend/day timing doesn't rescue it, size still drives the call |
| Zero-product edge case (invalid input) | Not Cancelled | 0.1% | ✅ Doesn't crash, degrades sensibly instead of erroring out — a real robustness pass |

**Bottom line:** this model is not equally risky to trust everywhere. It's a genuinely reliable flag for small, simple orders, and a genuinely poor one for large, varied orders — precisely the orders where a mistaken cancellation would cost the business the most.

## 6. A real, testable mitigation — not just a complaint

Finding a blind spot is only half the job. Here's one concrete, cheap thing to actually try: **lowering the decision threshold** (the cutoff probability where we call something "Cancelled") from the default 0.5.

**Analogy:** right now the model only raises its hand when it's more than 50% sure. Lowering that bar means it raises its hand more often — catching more real cancellations, but also crying wolf more often. This is a genuine trade-off, not a free fix.

In [ ]:
for threshold in [0.50, 0.35, 0.25, 0.15]:
    adjusted_pred = (results['confidence'] >= threshold).astype(int)
    cancelled = results[results['actual']==1]
    not_cancelled = results[results['actual']==0]
    recall = (adjusted_pred[cancelled.index]==1).mean()
    false_alarm_rate = (adjusted_pred[not_cancelled.index]==1).mean()
    print(f"Threshold {threshold:.2f}:  recall={recall:.1%}   false alarm rate={false_alarm_rate:.1%}")

**Reading this table:** dropping the threshold from 0.50 to 0.25 lifts recall from 86.6% to 92.2% — genuinely catching more real cancellations — but the false alarm rate rises from 7.9% to 11.1% too. **Whether that trade is worth it depends on what a false alarm actually costs the business** versus what a missed cancellation costs — a decision for the business, not something the model can decide on its own. This is the honest, correct way to respond to a discovered blind spot: quantify the trade-off, don't pretend there's a free fix.

## Summary — the failure catalog

| Finding | Evidence |
|---|---|
| Model has learned "small order = risky, big order = safe" | Missed cancellations: median 4 items/£113. Caught cancellations: median 1 item/£15 |
| It's confidently wrong, not narrowly unsure | Median confidence on missed cancellations: ~0.20, not close to the 0.5 cutoff |
| Rare countries add a small, secondary risk | Recall ~77% on long-tail countries vs. ~87% on the top 10 |
| Robust to a nonsensical/empty input | Zero-product edge case doesn't crash, returns a sane low-risk score |
| A real, testable mitigation exists | Lowering the threshold to 0.25 lifts recall to 92.2% at the cost of a higher false alarm rate |

**What I'd do next:** the real fix isn't threshold-tuning alone — it's adding a feature the model currently has no access to, like a customer's own order-size history, so a "large order" can be judged as large *for that customer* rather than in absolute terms. That's a bigger change than this project's scope, but it's the direction a genuine fix would go.
